In [1]:
import pandas as pd
import joblib
import sys
from pathlib import Path

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


In [2]:
RUN_TAG = "v1"
ARTIFACT_DIR = Path("../artifacts/model_search") / RUN_TAG

SUMMARY_PATH = ARTIFACT_DIR / "cv_summary.csv"
SEARCH_RESULTS_PATH = ARTIFACT_DIR / "search_results.joblib"

summary_df = pd.read_csv(SUMMARY_PATH)
search_results = joblib.load(SEARCH_RESULTS_PATH)

summary_df.head()

Loaded summary: ../artifacts/model_search/v1/cv_summary.csv -> shape=(5, 7)
Loaded search results: ../artifacts/model_search/v1/search_results.joblib -> models=['logreg', 'svc_rbf', 'knn', 'rf', 'hgb']


,model,best_score_refit_f2,best_mean_test_recall,best_mean_test_precision,best_mean_test_f2,best_mean_test_pr_auc,best_params
0,svc_rbf,0.916118,0.944924,0.816951,0.916118,0.921851,"{'clf__C': np.float64(72.86653737491046), 'clf__gamma': np.float64(0.026619018884890575)}"
1,rf,0.900404,0.925349,0.813167,0.900404,0.923146,"{'clf__max_depth': 10, 'clf__n_estimators': 220}"
2,knn,0.899316,0.922166,0.818403,0.899316,0.915006,"{'clf__n_neighbors': 30, 'clf__weights': 'distance'}"
3,hgb,0.889750,0.898952,0.855415,0.889750,0.931531,"{'clf__learning_rate': np.float64(0.052466345336252856), 'clf__max_leaf_nodes': 21}"
4,logreg,0.873978,0.919436,0.729929,0.873978,0.810868,{'clf__C': np.float64(4.5705630998014515)}


In [3]:
KOI_train_set = pd.read_csv("../data/processed/KOI_train_set.csv")
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
X_train = KOI_train_set.drop(columns=["label", "group_id"])
y_train = KOI_train_set["label"]
group_id = KOI_train_set["group_id"]

In [5]:
X_test = KOI_test_set.drop(columns=["label", "group_id"])
y_test = KOI_test_set["label"]

In [6]:
X_k2p = K2P_set.drop(columns=["label", "group_id"])
y_k2p = K2P_set["label"]

We will use the tuned models from `search_results.joblib` as a candidate pool for 3 model profiles:
* Best F2 model
* Best Recall model (with a minimum precision floor)
* Best Precision model (with a minimum recall floor)